# Exploración del CSV

**Responsable:** Juan Arias

**Objetivo:** conocer el CSV de metadatos antes de cargarlo y registrar sus inconsistencias, sin modificar los datos. La exploración del FASTA está en `00_exploracion.ipynb` (Thomas).

Modelo de referencia: `../MODELO.md`

In [1]:
from pathlib import Path

import pandas as pd

CSV = Path("../data/PanTEon_Database_metadata_v1.6.1.csv")
RESULTADOS = Path("../resultados")

df = pd.read_csv(CSV, encoding="utf-8-sig")
print(df.shape)
df.head()

(2790, 13)


,Kingdom,Phylum,Class,Order,Family,Species,Library?,Source,Source 2,Notes,Selected,TE_count,MCH_Parameters
0,Fungi,Basidiomycota,Agaricomycetes,Polyporales,Podoscyphaceae,Abortiporus biennis,Complete,RM2+MCH,NaN,B-GUT,RM2+MCH,1,"x=0, e=500, c=1"
1,Fungi,Mucoromycota,Mucoromycetes,Mucorales,Cunninghamellaceae,Absidia caerulea,Complete,RM2+MCH,NaN,B-GUT,RM2+MCH,10,"x=0, e=500, c=1"
2,Fungi,Mucoromycota,Mucoromycetes,Mucorales,Cunninghamellaceae,Absidia glauca,Complete,RM2+MCH,NaN,B-GUT,RM2+MCH,1,"x=0, e=500, c=1"
3,Fungi,Mucoromycota,Mucoromycetes,Mucorales,Cunninghamellaceae,Absidia repens,Complete,RM2+MCH,NaN,B-GUT,RM2+MCH,6,"x=0, e=500, c=1"
4,Animalia,Echinodermata,Asteroidea,Valvatida,Acanthasteridae,Acanthaster planci,complete,RM2+MCH,NaN,"ENSEMBL ""2023""",RM2+MCH,71,"x=10, e=500, c=1"


## 1. Columnas

Quedarse con las 9 columnas del enunciado: Kingdom, Phylum, Class, Order, Family, Species, Source, Source 2, Notes.

In [2]:
CONSERVAR = ["Kingdom", "Phylum", "Class", "Order", "Family", "Species", "Source", "Source 2", "Notes"]
TAXONOMIA = ["Kingdom", "Phylum", "Class", "Order", "Family"]


def limpiar(s):
    """strip + espacios dobles; los faltantes siguen faltando."""
    return s.str.strip().str.replace(r"\s+", " ", regex=True)


print("Filas:", len(df), "| Columnas:", df.shape[1])
print("Faltan del enunciado:", [c for c in CONSERVAR if c not in df.columns])
print("Se descartan:", [c for c in df.columns if c not in CONSERVAR])

Filas: 2790 | Columnas: 13
Faltan del enunciado: []
Se descartan: ['Library?', 'Selected', 'TE_count', 'MCH_Parameters']


**Conclusión:** 2.790 filas y 13 columnas. Las 9 columnas del enunciado están todas. Se descartan `Library?`, `Selected`, `TE_count` y `MCH_Parameters`; `TE_count` se compara antes con el FASTA (sección 6).

## 2. Espacios sobrantes y valores escritos de formas distintas en la taxonomía

In [3]:
espacios = pd.DataFrame({
    c: {
        "cambian_con_strip": (df[c].dropna() != df[c].dropna().str.strip()).sum(),
        "espacios_dobles": df[c].dropna().str.contains("  ").sum(),
    }
    for c in CONSERVAR
}).T
print(espacios)
for c in TAXONOMIA:
    s = df[c].dropna()
    raros = s[s != s.str.strip()].unique()
    if len(raros):
        print(c, [repr(v) for v in raros])

          cambian_con_strip  espacios_dobles
Kingdom                   0                0
Phylum                    1                0
Class                     1                0
Order                     9                0
Family                    2                0
Species                   0                0
Source                    0                0
Source 2                  0                0
Notes                     0                0
Phylum ["' Ascomycota'"]
Class ["' Gastropoda'"]
Order ["'Caryophyllales '", "'Sapindales '", "'Myrtales '", "'Rosales '", "'Fabales '", "'Asterales '", "'Ericales '", "'Alismatales '"]
Family ["'Fabaceae '", "'Araceae '"]


**Conclusión:** 13 valores cambian con `strip`, todos en la taxonomía (Phylum 1, Class 1, Order 9, Family 2), p. ej. `' Ascomycota'` y `'Rosales '`. No hay espacios dobles y `Species` está limpia. Sin `strip`, `'Rosales '` y `'Rosales'` contarían como dos órdenes.

## 3. Valores faltantes por columna

In [4]:
faltantes = df[CONSERVAR].isna().sum().to_frame("faltantes")
faltantes["%"] = (faltantes["faltantes"] / len(df) * 100).round(1)
faltantes

,faltantes,%
Kingdom,0,0.0
Phylum,3,0.1
Class,49,1.8
Order,29,1.0
Family,76,2.7
Species,0,0.0
Source,0,0.0
Source 2,2740,98.2
Notes,6,0.2


**Conclusión:** `Kingdom`, `Species` y `Source` están completas. Faltan Phylum 3, Class 49, Order 29, Family 76 y Notes 6. `Source 2` está vacía en 2.740 filas (98,2 %). Además de los NaN hay rellenos escritos como texto (sección 4).

## 4. Valores tipo "No order…" e *incertae sedis*

In [5]:
relleno = {}
for c in TAXONOMIA:
    s = limpiar(df[c].dropna())
    hits = s[s.str.match(r"(?i)^(no|not)\s")]
    if len(hits):
        relleno[c] = hits.value_counts()
for c, vc in relleno.items():
    print(f"{c}: {vc.sum()} filas")
    print(vc.to_string(), end="\n\n")

Order: 26 filas
Order
No trobat                                       7
No order                                        5
Not order identified Mamalia                    3
No ordre NCBI                                   2
No Orden No trobat                              2
Not order identified Enteropneusta              2
No ordre NCBI Caenogastropoda incertae sedis    1
No order (eupecaria incertae sedis)             1
Not order identified Testudines                 1
Not order identified Actinopteri                1
No order Dothideomycetes incertae sedis         1



**Conclusión:** solo `Order` tiene rellenos: 26 filas en 11 variantes, en catalán (`No trobat`, `No ordre NCBI`), español (`No Orden No trobat`) e inglés (`No order`, `Not order identified Mamalia`). Tres traen además *incertae sedis*. Falta decidir si pasan a `null`, que es lo coherente con «puede faltar» en MODELO.md.

## 5. Especies duplicadas (exactas y después de normalizar)

In [6]:
norm = limpiar(df["Species"]).str.lower()
print("Duplicados exactos:", df["Species"].duplicated(keep=False).sum())
print("Duplicados normalizados:", norm.duplicated(keep=False).sum())

Duplicados exactos: 0
Duplicados normalizados: 0


**Conclusión:** 0 duplicados, exactos o después de normalizar (strip + espacios dobles + minúsculas). `Species` sirve como llave única.

## 5b. Filas cuya `Species` no es una especie

In [7]:
no_especie = df[df["Notes"].str.contains("(?i)no especie", na=False)]
print("Marcadas en Notes:", len(no_especie), no_especie["Notes"].value_counts().to_dict())
print(no_especie["Species"].tolist())

# Casos similares sin marca: una sola palabra (taxón de rango superior)
una_palabra = df[~df["Species"].str.contains(" ") & ~df.index.isin(no_especie.index)]
print("\nUna palabra sin marca:", len(una_palabra), una_palabra["Species"].tolist())

# No binomiales: subespecies, cepas, f. sp., híbridos, nombres comunes
binomial = r"^[A-Z][a-z-]+ [a-z][a-z-]+$"
otros = df[~df["Species"].str.match(binomial) & ~df.index.isin(no_especie.index)]
print("\nNo binomiales sin marca:", len(otros))
otros[["Species", "Notes"]].head(20)

Marcadas en Notes: 44 {'Dfam no especie': 34, 'No especie Repbase': 10}
['Afrotheria', 'Amniota', 'Artiodactyla', 'Boreoeutheria', 'Carnivora', 'Catarrhini', 'Chiroptera', 'Chrysemys', 'Danio', 'Danionidae', 'Durocryptodira', 'Euarchontoglires', 'Euteleostomi', 'Eutheria', 'Glires', 'Haplorrhini', 'Heliconiini', 'Heliconius', 'Hominidae', 'Homininae', 'Hominoidea', 'Mammalia', 'Metatheria', 'Metazoa', 'Monotremata', 'Muridae', 'Murinae', 'Myomorpha', 'Perissodactyla', 'Platyrrhini', 'Primates', 'Protostomia', 'Rattus', 'Rodentia', 'Sauropsida', 'Scandentia', 'Sciuromorpha', 'Simiiformes', 'Strepsirrhini', 'Teleostei', 'Testudines', 'Testudinoidea', 'Tetrapoda', 'Xenarthra']

Una palabra sin marca: 1 ['Muroidea']

No binomiales sin marca: 52


,Species,Notes
165,Antilocapra americana peninsularis,Curated Dfam
224,Argopecten irradians concentricus,Curated Dfam
250,Ascobolus immersus rn42,"ENSEMBL ""2023"""
471,Basidiobolus meristosporus cbs 93173,"ENSEMBL ""2023"""
487,Big-eye mandarin fish,"ENSEMBL ""2023"""
507,Blumeria graminis f sp tritici,"ENSEMBL ""2023"""
512,Boehmeria nivea var tenacissima,"ENSEMBL ""2025"""
553,Branchiostoma floridae x branchiostoma belcheri,"ENSEMBL ""2023"""
612,Canis lupus familiaris,Release 1.2
633,Caryopteris x clandonensis,"ENSEMBL ""2025"""


**Conclusión:** las 44 filas marcadas en `Notes` (`Dfam no especie` 34, `No especie Repbase` 10) son taxones superiores (Amniota, Carnivora, Danio…). `Muroidea` es otro taxón superior pero no tiene marca. Además hay 52 nombres no binomiales sin marca: subespecies, cepas (`Ascobolus immersus rn42`), *f. sp.*, híbridos con `x` y algún nombre común (`Big-eye mandarin fish`). Falta decidir si las 44 se cargan con una marca o se excluyen, y si los no binomiales se aceptan tal cual.

## 5c. Valores distintos de `Source`, `Source 2` y `Notes`

In [8]:
for c in ["Source", "Source 2", "Notes"]:
    print(df[c].value_counts(dropna=False).to_string(), end="\n\n")

FUENTES = {"Repbase", "Dfam", "RM2+MCH"}
no_fuente = df[df["Source 2"].notna() & ~df["Source 2"].isin(FUENTES)]
no_fuente[["Species", "Source 2"]]

Source
RM2+MCH    2404
Dfam        386

Source 2
NaN                          2740
Repbase                        45
Genero coincidente              2
Genero y familia repetido       1
Obtained by Johan P.            1
Dfam                            1

Notes
B-GUT                 1479
ENSEMBL "2023"         375
ENSEMBL "2025"         373
Curated Dfam           293
Uncurated Dfam         165
Release 1.2             43
Dfam no especie         34
APTEdb                  12
No especie Repbase      10
NaN                      6



,Species,Source 2
573,Burhinus oedicnemus,Genero y familia repetido
703,Chironomus riparius,Genero coincidente
704,Chironomus tepperi,Genero coincidente
892,Cucumis sativus,Obtained by Johan P.


**Conclusión:** `Source` solo tiene `RM2+MCH` (2.404) y `Dfam` (386). `Source 2` tiene 50 valores: 45 `Repbase` y 1 `Dfam` son fuentes; 4 son comentarios (`Genero coincidente` ×2, `Genero y familia repetido`, `Obtained by Johan P.`). `Notes` mezcla origen o versión (`B-GUT`, `ENSEMBL "2023"`, `Curated Dfam`) con la marca `no especie`.

## 5d. Taxonomías sospechosas

In [9]:
tax = df[TAXONOMIA + ["Species"]].apply(limpiar)
sospechosas = []
for hijo, padre in [("Phylum", "Kingdom"), ("Class", "Phylum"), ("Order", "Class"), ("Family", "Order")]:
    pares = tax.dropna(subset=[hijo, padre])
    pares = pares[~pares[hijo].str.match(r"(?i)^(no|not)\s")]
    for valor, grupo in pares.groupby(hijo):
        conteo = grupo[padre].value_counts()
        # el padre minoritario es el sospechoso
        for p in conteo.index[1:]:
            for sp in grupo.loc[grupo[padre] == p, "Species"]:
                sospechosas.append({"nivel": hijo, "valor": valor, "padre_mayoritario": conteo.index[0],
                                    "padre_raro": p, "species": sp})
sospechosas = pd.DataFrame(sospechosas)
print(len(sospechosas))
sospechosas

11


,nivel,valor,padre_mayoritario,padre_raro,species
0,Class,Marchantiopsida,Bryophyta,Marchantiophyta,Lunularia cruciata
1,Order,Passeriformes,Aves,Holothuroidea,Holothuria scabra
2,Order,Trichoptera,Insecta,Aves,Himantopus himantopus leucocephalus
3,Family,Andrenidae,Hymenoptera,Neoptera,Andrena haemorrhoa
4,Family,Chironomidae,Neoptera,Diptera,Chironomus tepperi
5,Family,Coelopidae,Diptera,Neoptera,Coelopa pilipes
6,Family,Sarcophagidae,Neoptera,Diptera,Sarcophaga variegata
7,Family,Sporocadaceae,Xylariales,Amphisphaeriales,Truncatella angustata
8,Family,Trichogrammatidae,Hymenoptera,Neoptera,Trichogramma pretiosum
9,Family,Yponomeutidae,Neoptera,Lepidoptera,Yponomeuta sedellus


**Conclusión:** son taxones que cuelgan de padres distintos (ya con `strip`). Los más claros: `Passeriformes` (aves) dentro de Holothuroidea (equinodermos) y `Trichoptera` (insectos) dentro de Aves. Además `Neoptera`, una infraclase, aparece como `Order` en varias familias de insectos; `Marchantiopsida` cuelga de Bryophyta y `Sporocadaceae` de dos órdenes. Solo se listan; aquí no se corrige nada.

## 6. `TE_count` frente al conteo real del FASTA (antes de descartarla)

Usar `resultados/especies_fasta.csv`, que genera el notebook de Thomas.

## 7. Cruce por especie

- Especies del FASTA que no están en el CSV
- Especies del CSV sin TEs en el FASTA

Normalizar ambos lados (espacios y mayúsculas) antes de comparar.

## 8. Inconsistencias y decisiones

| Problema | Casos | Decisión | Justificación |
|---|---|---|---|
| | | | |

## Reporte de calidad

In [10]:
reporte = pd.DataFrame([
    ("Espacios al inicio/final en la taxonomía", "'Rosales '", int(espacios["cambian_con_strip"].sum())),
    ("Taxonomía con algún NaN", "Family vacía", int(df[TAXONOMIA].isna().any(axis=1).sum())),
    ("Rellenos 'No …' en Order", "No trobat", int(sum(vc.sum() for vc in relleno.values()))),
    ("Species que no es especie (marcada en Notes)", no_especie["Species"].iloc[0], len(no_especie)),
    ("Species no binomial sin marca", otros["Species"].iloc[0], len(otros)),
    ("Source 2 con texto que no es fuente", no_fuente["Source 2"].iloc[0], len(no_fuente)),
    ("Taxonomía sospechosa (padre minoritario)", "Passeriformes en Holothuroidea", len(sospechosas)),
    ("Species duplicada tras normalizar", "-", int(norm.duplicated(keep=False).sum())),
], columns=["Problema", "Ejemplo", "Filas afectadas"])

filas = ["| Problema | Ejemplo | Filas afectadas |", "| --- | --- | --- |"]
filas += [f"| {p} | `{e}` | {n} |" for p, e, n in reporte.itertuples(index=False)]
RESULTADOS.mkdir(exist_ok=True)
(RESULTADOS / "reporte_calidad_csv.md").write_text(
    "# Reporte de calidad del CSV de metadatos\n\n"
    "Generado por `notebooks/00_exploracion_csv.ipynb`. No se modificó ningún dato.\n\n"
    + "\n".join(filas) + "\n",
    encoding="utf-8",
)
reporte

,Problema,Ejemplo,Filas afectadas
0,Espacios al inicio/final en la taxonomía,'Rosales ',13
1,Taxonomía con algún NaN,Family vacía,122
2,Rellenos 'No …' en Order,No trobat,26
3,Species que no es especie (marcada en Notes),Afrotheria,44
4,Species no binomial sin marca,Antilocapra americana peninsularis,52
5,Source 2 con texto que no es fuente,Genero y familia repetido,4
6,Taxonomía sospechosa (padre minoritario),Passeriformes en Holothuroidea,11
7,Species duplicada tras normalizar,-,0
